In [7]:
import sys, os, cv2
origin='C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/'

sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Extract_Data')
from Read import Vision
from google.cloud import vision
from google.cloud.vision_v1 import types

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = origin+'Tokyo_Jobs\\Environment\\GoogleVision\\practice-302516-01e0f7245b03.json'
# Instantiates a client
client = vision.ImageAnnotatorClient()

In [197]:
sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\General')
from PageCut import HoriPart
from OCR import Clova
api_url='https://1srlcnzf08.apigw.ntruss.com/custom/v1/2412/9a859f9b3a7d952aad17f388d1a445041f8aef0f1eccc6fcce8d3a856272fcbd/general'
secret_key='eEhyV0NGRlRLVXpZVWZnWFNDamRVaWFYZ1NSQ1NKSFI='


In [198]:
Year,Showa='1942','17'
Quality='HQ'

In [330]:
Page=60

file_path2='Tokyo_Jobs/Raw_Data/Metropolitan_DA/'+str(Year)+'/'+str(Quality)+'/Page'+'{:03d}'.format(Page)+'.jpg'
file_path3='C:/Users/Keitaro Ninomiya/Desktop/Page020.png'
img=cv2.imread(os.path.join(origin,file_path2))
textsJA=Vision(img,'ja',client)
textsZH=Vision(img,'zh',client)
textsZH1=Vision(img,'zh-Hans',client)
textsZH2=Vision(img,'zh-Hant',client)
textsZH3=Vision(img,'zh-Hant-HK',client)
textsCLOVA=Clova(origin,Page,api_url,secret_key,Year,Quality)

In [331]:
img=cv2.imread(os.path.join(origin,file_path2))
copy=img.copy()
copy2=img.copy()
copy3=img.copy()
copyNDL=img.copy()
copyCV=img.copy()

In [332]:
def draw_ocr_resultsNDL(image, textNDL, color=(0, 255, 0)):
	# unpacking the bounding box rectangle and draw a bounding box
	# surrounding the text along with the OCR'd text itself
    for element in textNDL:
        startX,startY,endX,endY = element[0],element[1],element[2],element[3]
        cv2.rectangle(image, (startX, startY), (endX, endY), color, 2)
    	# return the output image
    return image

In [333]:
def draw_ocr_resultsCLOVA(image, textsCLOVA, color=(0, 255, 0)):
	# unpacking the bounding box rectangle and draw a bounding box
	# surrounding the text along with the OCR'd text itself
    for element in textsCLOVA:
        rect=element['boundingPoly']
        startX,startY,endX,endY = int(rect['vertices'][0]['x']),int(rect['vertices'][0]['y']),int(rect['vertices'][2]['x']),int(rect['vertices'][2]['y'])
        cv2.rectangle(image, (startX, startY), (endX, endY), color, 2)
    	# return the output image
    return image

In [335]:
import json
f = open(origin+'Tokyo_Jobs/Processed_Data/'+str(Year)+'/NDL/'+'Page{:03d}'.format(Page)+'.json', encoding="utf8")
data = json.load(f)[0]

AnnotatedImageJA=draw_ocr_results(img, textsJA, color=(0, 255, 0))
cv2.namedWindow("Resized_WindowJA", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Resized_WindowJA", 1280, 1440)
cv2.imshow('Resized_WindowJA',AnnotatedImageJA)
cv2.waitKey(0)

AnnotatedImageZH1=draw_ocr_results(copy, textsZH, color=(0, 255, 0))
cv2.namedWindow("Resized_WindowZH1", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Resized_WindowZH1", 1280, 1440)
cv2.imshow('Resized_WindowZH1',AnnotatedImageZH1)
cv2.waitKey(0)

AnnotatedImageCLOVA=draw_ocr_resultsCLOVA(copyCV, textsCLOVA, color=(0, 255, 0))
cv2.namedWindow("Resized_WindowCV", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Resized_WindowCV", 1280, 1440)
cv2.imshow('Resized_WindowCV',AnnotatedImageCLOVA)
cv2.waitKey(0)

AnnotatedImageNDL=draw_ocr_resultsNDL(copyNDL, data, color=(0, 255, 0))
cv2.namedWindow("Resized_WindowNDL", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Resized_WindowNDL", 1280, 1440)
cv2.imshow('Resized_WindowNDL',AnnotatedImageNDL)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [323]:
Desktop='C:/Users/Keitaro Ninomiya/Desktop'

cv2.imwrite(os.path.join(Desktop,'OCR_JA.png'), AnnotatedImageJA)
cv2.imwrite(os.path.join(Desktop,'OCR_ZH.png'), AnnotatedImageZH1)
cv2.imwrite(os.path.join(Desktop,'OCR_CLOVA.png'), AnnotatedImageCLOVA)
cv2.imwrite(os.path.join(Desktop,'OCR_NDL.png'), AnnotatedImageNDL)

True

In [224]:
data[0]

[2098, 430, 2531, 485, '四日']

In [162]:
def draw_ocr_results(image, texts, color=(0, 255, 0)):
	# unpacking the bounding box rectangle and draw a bounding box
	# surrounding the text along with the OCR'd text itself
    for element in texts:
        rect=element.bounding_poly
        startX,startY,endX,endY = rect.vertices[0].x,rect.vertices[0].y,rect.vertices[2].x,rect.vertices[2].y
        cv2.rectangle(image, (startX, startY), (endX, endY), color, 2)
    	# return the output image
    return image

def Center(PolyBox):
    ymax,ymin=min([d.y for d in PolyBox]),max([d.y for d in PolyBox])
    xmax,xmin=min([d.x for d in PolyBox]),max([d.x for d in PolyBox])
    return({'YCenter':(ymax+ymin)//2,'XCenter':(xmax+xmin)//2})

def ToDict(Box):
    Output={}
    Output['description']=Box.description
    Output['bounding_poly']=[d for d in Box.bounding_poly.vertices]
    Output['Center']=Center(Output['bounding_poly'])
    return(Output)

In [186]:
HH,WW=img.shape[:2]

DictBox=[ToDict(d) for d in texts[1:]]
Box=[d for d in DictBox if d['Center']['YCenter']<HH//2]

In [189]:
texts[0].description

'中野區出張所\n杉並區出張所:\n登島區出張所···\n彥野川區出張所:\n荒川區出張所\n王子槭出張所\n板橋區出張所\n二:\n你注南通\n金門第三關門\n法立區出效辦\n向島區出張所\n根基城主食食\n日專主義社\n葛飾區出張所\n江戶川區出張所:\n土木試驗所\n湖區\n電氣 局\n總務\n好····\n小\n經理:\nUNN\n…\n色龍\nCO\n繼··\n…\n業務\n事業普及評\n保線 …\n交通鋼調查課..…\n電車營業所\n自動車營業所 ..\n社\n德····\n電力…\n營業所\n市民動員布\n··\n…殺蝦\n診療所\n藏\n***\n…\n業業樂………\n辉南社\n海\n……\n國民精神總動員課\n防衡課……\n軍事接護課…\n倩掃部:\n作業課\n沿海部:\n技術\n…\n地所碟…\n褸務所:\n中央卸喪市場.\n整育院…\n庶務課…\nCON\nwww MA\n社\n***\n...\n***\n...\n***\n***\nw\n2\nwww\n***\n***\n***\n***\n***\n...\nAST\n***\nw\n***\n***\nimg\n***IRB\n012年\n2012-\n***\nw\n124\n128\n张\n18%\n129.\nwww.\n122\nTel\n2\n122\n20\n花\n*** 1101\ndoa\nHOB...\n路\nHOL...\nHOLL...\n1104\n2010···\n···110元\n1111\n**** ***11110\n...11\n一切知道\nHE ***1151\nXIB...\n211...\n15...\nSEL...\n10.115\n***1150\n版\n**19\n---\nERB...\nOXII....\n01....\n牌\n***KE\nAND\n****\nMAX\nTRA\n... Re\n... K\n>\nBall\nMaR ...\n*** BEM\nBER\n9211***\nfiks\n220\n628 ***\nz\n220 ...\n结果10\n112\n***1/20\n東京都立中央図書館\nRST/0317/88/G'

In [164]:
import img2pdf
from PIL import Image
from pdfreader import PDFDocument, SimplePDFViewer

# opening image
image = Image.open(os.path.join(origin,file_path2))
 
# converting into chunks using img2pdf
pdf_bytes = img2pdf.convert(image.filename)


In [172]:
Box[0]['bounding_poly'][0]

x: 1058
y: 161

In [180]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4, portrait
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
import webbrowser


# 白紙をつくる（A4縦）
FILENAME = 'HelloWorld.pdf'
c = canvas.Canvas(FILENAME, pagesize=(WW, HH))

# フォント登録
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.cidfonts import UnicodeCIDFont
pdfmetrics.registerFont(UnicodeCIDFont('HeiseiMin-W3', isVertical=True))
c.setFont('HeiseiMin-W3', 7)
font_size=2

# 真ん中に文字列描画
for text in Box:
    c.drawCentredString(text['bounding_poly'][2].x, HH-text['bounding_poly'][2].y, text['description'])

# Canvasに書き込み
c.showPage()
# ファイル保存
c.save()

# ブラウザーで表示
webbrowser.open(FILENAME)

True

In [205]:
AnnotatedImage=draw_ocr_results(img, texts[1:], color=(0, 255, 0))
cv2.namedWindow("Resized_Window", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Resized_Window", 1280, 1440)
cv2.imshow('Resized_Window',AnnotatedImage)
cv2.waitKey(0)
cv2.destroyAllWindows()
